# Manual trading Round 1

Simulate the auction for Mushrooms, assuming our order gets executed after those with higher prices and already in the book (price-time priority)

In [3]:
def simulate_auction(my_bid_price, my_bid_volume, terminal_value=20, fee=0.05):
    # Data from the Ember Mushroom stale book
    # Sorted Asks: (Price, Volume) - Ascending
    asks = [[12, 20000], [13, 25000], [14, 35000], [15, 6000], [16, 5000], [18, 10000], [19, 12000]]
    
    # Sorted Bids: (Price, Volume, is_me) - Descending
    # Note: I'm adding 'is_me' to track your specific fill
    bids = [
        [20, 43000, False],
        [19, 17000, False],
        [18, 6000, False],
        [17, 5000, False],
        [16, 10000, False],
        [15, 5000, False],
        [14, 10000, False],
        [13, 7000, False]
    ]
    
    # Add your order to the list
    bids.append([my_bid_price, my_bid_volume, True])
    # Re-sort bids by price (descending). 
    # Python's sort is stable, so you'll be behind existing orders at the same price.
    bids.sort(key=lambda x: x[0], reverse=True)

    total_my_fill = 0
    
    # Match engine
    for ask in asks:
        ask_price, ask_vol = ask
        for bid in bids:
            bid_price, bid_vol, is_me = bid
            
            if bid_vol <= 0 or ask_vol <= 0:
                continue
                
            if bid_price >= ask_price:
                match_vol = min(bid_vol, ask_vol)
                
                if is_me:
                    total_my_fill += match_vol
                
                # Deduct volume from the book
                bid[1] -= match_vol
                ask_vol -= match_vol
        
    profit = total_my_fill * (terminal_value - my_bid_price - fee)
    return total_my_fill, profit

# Testing different scenarios
prices_to_test = [14, 15, 16, 17, 18, 19, 20, 21]
print(f"{'Price':<8} | {'Fill Volume':<12} | {'Net Profit'}")
print("-" * 40)
for p in prices_to_test:
    fill, profit = simulate_auction(p, 100000) # Assuming you want as much as possible
    print(f"{p:<8} | {fill:<12,.0f} | {profit:,.2f}")

Price    | Fill Volume  | Net Profit
----------------------------------------
14       | 0            | 0.00
15       | 0            | 0.00
16       | 10,000       | 39,500.00
17       | 20,000       | 59,000.00
18       | 35,000       | 68,250.00
19       | 53,000       | 50,350.00
20       | 70,000       | -3,500.00
21       | 100,000      | -105,000.00
